# F1 (YOLO26s-OBB) Training Orchestrator

This notebook performs sparse repository cloning, installs dependencies in editable mode, runs unit tests, and executes an F1 training run on Dual Tesla T4 GPUs.

One run per ablation condition, selected with `--condition`:

| Condition | Dataset | Augmentation |
|---|---|---|
| `c1` | Raw 640×360 images | Minimal (geometric only) |
| `c2` | Raw 640×360 images | Classic YOLO (mosaic, mixup, copy-paste) |
| `c3` | LaMa-cleaned images | Minimal, identical to `c1` |

`c1` and `c3` share every hyperparameter by construction: the only difference between them is the pixel content of the training images, which is what the intra-family gain of the study measures.

### Key Features:
- **Per-run isolation**: each condition writes to its own output directory and its own Google Drive subfolder (`f1_c1`, `f1_c2`, `f1_c3`), so concurrent runs never overwrite each other's checkpoints.
- **Checkpoint Resume**: downloads this run's `last.pt` from its Drive subfolder to resume an interrupted Kaggle session, discarding the file if its size does not match.
- **Incremental Sync**: synchronizes `results.csv`, `best.pt` and `last.pt` to Drive every 5 epochs, from the primary process only.
- **Fast Dev Run Mode**: `--fast-dev-run` executes a smoke test (1 epoch on 1% of the dataset) before launching a full run.

## 1. Clone Repository & Install Package Dependencies

In [ ]:
import os
from pathlib import Path

REPO_NAME = 'ia_article'
REPO_URL = 'https://github.com/unsa-semester-2026-A/ia_article.git'
BRANCH_NAME = '12-training-base-1'

# --- Environment Detection: Kaggle vs Colab ---
BASE_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
%cd {BASE_DIR}

REPO_PATH = Path(BASE_DIR) / REPO_NAME

# 1. Sparse clone repository
if not REPO_PATH.exists():
    print(f"Cloning {REPO_NAME} (branch {BRANCH_NAME})...")
    !git clone -q --depth 1 --branch {BRANCH_NAME} --filter=blob:none --sparse {REPO_URL}
    %cd {REPO_NAME}
    !git sparse-checkout set experiments
    %cd experiments
else:
    print(f"Updating existing {REPO_NAME} repository...")
    %cd {REPO_NAME}
    !git checkout {BRANCH_NAME}
    !git pull -q origin {BRANCH_NAME}
    %cd experiments

# 2. Verify working directory
current_dir = Path(os.getcwd())
if current_dir.name != 'experiments':
    raise RuntimeError(f"Directory navigation failed. Current path: {current_dir}")

# 3. Install package in editable mode
print("Installing package in editable mode with [cloud] dependencies...")
%pip install -q -e .[cloud]


## 2. Run Unit Tests

Execute co-located unit tests (BaseTrainingPipeline and Base1Trainer) to ensure system stability before launching training.

In [ ]:
# Run unit tests to verify training pipeline integrity
!pytest src/training/ -v


## 3. Execute the Training Run

### Option A: Fast Dev Run (Smoke Test — ~30 seconds)
Runs 1 single epoch on 1% of the dataset to verify GPU acceleration, data loading, and Google Drive sync.

### Option B: Full Production Training
Cap of 40 epochs on Dual Tesla T4, with early stopping at `patience=5`. The pilot run reached its best mAP50-95 around epoch 6 and the following 33 epochs added ~0.01, so the expected stop is between epochs 11 and 16, or roughly 3 hours at ~14.7 min per epoch.

Change `--condition` to launch `c2` or `c3`.

In [ ]:
# Option A: Fast Smoke Test (Uncomment to run a 30-second dry run)
# !python -m src.training.trainers.train_base_1 --condition c1 --fast-dev-run


In [ ]:
# Option B: Full Production Training (cap 40 epochs, patience 5, Drive resume per run)
!python -m src.training.trainers.train_base_1 --condition c1
